# Qwen3.5-4B Base: S2 direction comparison

Companion to `euphoria-init.ipynb`. Run in a **fresh kernel** on the same WSL machine; the two 4B checkpoints should not remain loaded together on the 16 GB GPU. This notebook uses the same source-paper S2 prompts and fixed raw decoder block 30. It compares Base with the saved post-trained results if those files are present.

**Scope:** the source-prompt scores and suffix factorial below are exploratory transfer checks, not the fresh-prompt confirmation promised by the frozen motif in `euphoria-init.ipynb`. Do not revise that prediction after seeing these results. Run cells top to bottom. Keep `pain-axis-source` at the recorded source commit.

In [ ]:
import json
import subprocess
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B-Base"
SOURCE_COMMIT = "8d1649c03a63a39c9aa092532c376800cc4a3863"
BLOCK = 30  # zero-indexed raw decoder block, frozen from the post-trained run
BATCH_SIZE = 4
run_dir = Path("runs/qwen35-4b-base-s2")
run_dir.mkdir(parents=True, exist_ok=True)

source_repo = Path("pain-axis-source")
if not source_repo.exists():
    subprocess.run(["git", "clone", "https://github.com/valen-research/Pain-axis.git", str(source_repo)], check=True)
actual_commit = subprocess.check_output(
    ["git", "-C", str(source_repo), "rev-parse", "HEAD"], text=True
).strip()
assert actual_commit == SOURCE_COMMIT, f"Source changed: {actual_commit}; expected {SOURCE_COMMIT}"

data = json.loads((source_repo / "datasets/3.1_pain_and_control_datasets.json").read_text())
rows = data["datasets"]["S2_1P"]["sentences"]
y = np.array([r["category"].startswith("A") for r in rows], dtype=bool)
groups = np.array([r["set"] for r in rows])
assert len(rows) == 200 and Counter(y) == {True: 100, False: 100}
assert all(r["prompt"].endswith("I feel:") for r in rows)
assert len(set(groups)) == 20
assert all(Counter(y[groups == g]) == {True: 5, False: 5} for g in set(groups))
print("Source:", actual_commit, "| S2_1P:", len(rows), "| categories:", Counter(r["category"] for r in rows))

In [ ]:
hf = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device_map="auto")
hf.eval()
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = "left"
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

decoder_layers = [(name, module) for name, module in hf.named_modules()
                  if module.__class__.__name__ == "Qwen3_5DecoderLayer"]
assert len(decoder_layers) == 32
model_revision = getattr(hf.config, "_commit_hash", None)
print("Model:", MODEL_NAME, "| revision:", model_revision)
print("Blocks:", len(decoder_layers), "| hidden width:", getattr(hf.config, "text_config", hf.config).hidden_size)
print("Allocated VRAM (GiB):", round(torch.cuda.memory_allocated() / 2**30, 2))

## Extract S2 source activations

Use plain-text prompts as stored, left pad, and read the final `:` token. Hugging Face's last `hidden_states` entry is post-norm for this model, so capture the raw final block output by hook. All stored states are float32 CPU tensors shaped `[prompt, block, feature]`.

In [ ]:
states, token_ids = [], []
for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="S2 Base"):
    batch = rows[start:start + BATCH_SIZE]
    inputs = tok([r["prompt"] for r in batch], padding=True, return_tensors="pt").to(hf.device)
    token_ids.extend([ids[mask.bool()].tolist()
                      for ids, mask in zip(inputs["input_ids"], inputs["attention_mask"])])

    captured_last = {}
    def capture_last(module, module_inputs, output):
        captured_last["state"] = output[0] if isinstance(output, tuple) else output

    handle = decoder_layers[-1][1].register_forward_hook(capture_last)
    try:
        with torch.inference_mode():
            output = hf(**inputs, output_hidden_states=True, use_cache=False, return_dict=True)
    finally:
        handle.remove()

    assert len(output.hidden_states) == 33
    assert captured_last["state"].shape == output.hidden_states[-1].shape
    per_block = [h[:, -1, :].float().cpu() for h in output.hidden_states[1:]]
    per_block[-1] = captured_last["state"][:, -1, :].float().cpu()
    states.append(torch.stack(per_block, dim=1))

x = torch.cat(states, dim=0)
assert x.shape == (200, 32, 2560) and torch.isfinite(x).all()
assert all(tok.decode(ids[-1]) == ":" for ids in token_ids)
print("Raw block activations:", tuple(x.shape))

post_dir = Path("runs/qwen35-4b-posttrained-s2")
post_activation_path = post_dir / "activations_raw_blocks.pt"
if post_activation_path.exists():
    # Local file produced by euphoria-init.ipynb; torch.load contains prompt dictionaries.
    post_record = torch.load(post_activation_path, map_location="cpu", weights_only=False)
    assert [r["prompt"] for r in post_record["rows"]] == [r["prompt"] for r in rows]
    assert post_record["token_ids"] == token_ids, "Tokenization differs across checkpoints"
    print("Post-trained prompt text and token IDs match exactly")
else:
    print("Post-trained activation file absent; cross-check skipped")

torch.save({"model": MODEL_NAME, "model_revision": model_revision,
            "source_commit": actual_commit, "dataset": "S2_1P", "rows": rows,
            "token_ids": token_ids, "states": x,
            "state_convention": "raw decoder block outputs; final block captured by hook"},
           run_dir / "activations_raw_blocks.pt")

## Grouped held-out AUC

Use the same five folds grouped by source `set` as in the post-trained notebook. Fit both mean-difference and control PCA exclusively on each training fold. The best layer is descriptive (chosen on these folds); **block 30 remains fixed** for all transfer comparisons below.

In [ ]:
def fit_directions(pain, control):
    raw = pain.mean(0) - control.mean(0)
    centered = control - control.mean(0)
    eigenvalues, eigenvectors = torch.linalg.eigh(centered @ centered.T)
    order = torch.argsort(eigenvalues, descending=True)
    eigenvalues = eigenvalues[order].clamp_min(0)
    eigenvectors = eigenvectors[:, order]
    assert eigenvalues.sum() > 0
    fraction = eigenvalues.cumsum(0) / eigenvalues.sum()
    k = int(torch.searchsorted(fraction, 0.5).item()) + 1
    basis = centered.T @ eigenvectors[:, :k]
    basis /= eigenvalues[:k].clamp_min(1e-8).sqrt()[None, :]
    denoised = raw - basis @ (basis.T @ raw)
    return raw, F.normalize(denoised, dim=0), k

fold_results = []
for fold, (train, test) in enumerate(GroupKFold(n_splits=5).split(x, y, groups)):
    for block in range(x.shape[1]):
        pain = x[train, block][y[train]]
        control = x[train, block][~y[train]]
        raw, denoised, k = fit_directions(pain, control)
        for method, direction in (("raw", F.normalize(raw, dim=0)), ("denoised", denoised)):
            scores = (x[test, block] @ direction).numpy()
            fold_results.append({"fold": fold, "block": block, "method": method,
                                 "auc": roc_auc_score(y[test], scores), "removed_pcs": k})

cv = pd.DataFrame(fold_results)
cv.to_csv(run_dir / "grouped_s2_cv.csv", index=False)
summary = cv.groupby(["block", "method"])["auc"].mean().unstack()
print("Base top denoised blocks:")
print(summary.sort_values("denoised", ascending=False).head(8).round(4))
print("\nFixed block 30:")
print(summary.loc[[BLOCK]].round(4))

post_cv_path = post_dir / "denoised_direction_cv.csv"
if post_cv_path.exists():
    post_cv = pd.read_csv(post_cv_path)
    print("Post-trained fixed block 30 mean AUC:",
          round(post_cv.loc[post_cv.layer == BLOCK, "auc"].mean(), 4))

In [ ]:
raw_direction, base_direction, k = fit_directions(x[y, BLOCK], x[~y, BLOCK])
ref_scores = x[:, BLOCK] @ base_direction
ref_mean, ref_std = ref_scores.mean().item(), ref_scores.std(unbiased=False).item()
assert ref_std > 0
torch.save({"model": MODEL_NAME, "model_revision": model_revision,
            "source_commit": actual_commit, "dataset": "S2_1P", "block": BLOCK,
            "raw_direction": raw_direction, "denoised_direction": base_direction,
            "control_pcs_removed": k, "reference_mean": ref_mean,
            "reference_std": ref_std}, run_dir / "s2_direction_block30.pt")
print("Saved fixed block-30 Base direction | control PCs removed:", k)

## Source-category neighborhood and person × suffix check

These use the same source scenarios as the post-trained exploration. They compare models; they are **not** the held-out confirmation in the frozen motif. All z-scores use this Base checkpoint's S2 reference mean and population standard deviation. One 3P source row (`C2`, set 20) ends in `She feels:` and is excluded with its 1P pair from the suffix factorial.

In [ ]:
def score_prompts(prompts):
    scores = []
    for start in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Block 30", leave=False):
        batch = prompts[start:start + BATCH_SIZE]
        inputs = tok(batch, padding=True, return_tensors="pt").to(hf.device)
        with torch.inference_mode():
            output = hf(**inputs, output_hidden_states=True, use_cache=False, return_dict=True)
        states = output.hidden_states[BLOCK + 1][:, -1, :].float().cpu()
        scores.extend((states @ base_direction).tolist())
    return np.array(scores)

def to_z(scores):
    return (np.asarray(scores) - ref_mean) / ref_std

category = pd.DataFrame({"category": [r["category"] for r in rows],
                         "set": groups, "prompt": [r["prompt"] for r in rows],
                         "z_s2": to_z(ref_scores.numpy())})
category.to_csv(run_dir / "source_categories_block30.csv", index=False)
print(category.groupby("category")["z_s2"].mean().round(3).to_string())

control_records = []
for name in ("Numb_1P", "Arousal_1P", "Random_1P"):
    control_rows = data["datasets"][name]["sentences"]
    assert all(r["prompt"].endswith("I feel:") for r in control_rows)
    control_scores = to_z(score_prompts([r["prompt"] for r in control_rows]))
    for row, score in zip(control_rows, control_scores):
        control_records.append({"dataset": name, "category": row["category"],
                                "prompt": row["prompt"], "z_s2": score})
controls = pd.DataFrame(control_records)
controls.to_csv(run_dir / "neighborhood_block30.csv", index=False)
print("\nStandalone controls:")
print(controls.groupby("dataset")["z_s2"].agg(["count", "mean", "median"]).round(3))

In [ ]:
third_rows = data["datasets"]["S2_3P"]["sentences"]
third_by_key = {(r["category"], r["set"]): r for r in third_rows}
assert len(third_by_key) == len(rows) == 200
assert set(third_by_key) == {(r["category"], r["set"]) for r in rows}
third_paired = [third_by_key[(r["category"], r["set"])] for r in rows]

valid_indices = [i for i, row in enumerate(third_paired)
                 if row["prompt"].endswith("I feel:")]
assert len(valid_indices) == 199

factorial_rows = []
for idx in valid_indices:
    first, third = rows[idx], third_paired[idx]
    for story_person, source in (("1P", first), ("3P", third)):
        for suffix_person, suffix in (("I", "I feel:"), ("He", "He feels:")):
            prompt = source["prompt"].removesuffix("I feel:") + suffix
            factorial_rows.append({"category": first["category"], "set": first["set"],
                                   "group": "pain" if y[idx] else "control",
                                   "story_person": story_person, "suffix_person": suffix_person,
                                   "prompt": prompt})

factorial = pd.DataFrame(factorial_rows)
assert len(factorial) == 199 * 4
factorial["z_s2"] = to_z(score_prompts(factorial["prompt"].tolist()))
factorial.to_csv(run_dir / "story_person_x_suffix_block30.csv", index=False)
print(factorial.groupby(["group", "story_person", "suffix_person"])["z_s2"]
      .agg(["count", "mean"]).round(3))

wide = factorial.pivot(index=["category", "set", "group"],
                       columns=["story_person", "suffix_person"], values="z_s2")
wide["alignment_effect"] = (wide[("1P", "I")] + wide[("3P", "He")]
                            - wide[("1P", "He")] - wide[("3P", "I")])
by_set = (wide.reset_index().groupby(["set", "group"])["alignment_effect"]
          .mean().unstack())
effects = (by_set["pain"] - by_set["control"]).to_numpy()
rng = np.random.default_rng(20260923)
boot = rng.choice(effects, size=(10_000, len(effects)), replace=True).mean(axis=1)
print("\nAlignment by group:", wide.groupby(level="group")["alignment_effect"].mean().round(3).to_dict())
print("Extra pain alignment:", round(effects.mean(), 3),
      "| set bootstrap interval:", np.quantile(boot, [0.025, 0.975]).round(3),
      "| positive sets:", int((effects > 0).sum()), "/", len(effects))

## Cross-check against saved post-trained artifacts

Raw coordinates are compared only because these checkpoints share a model lineage and hidden width; a cosine or transferred vector in one model's residual basis is exploratory. Own-checkpoint scores below are in-sample because each direction was fitted on the full S2 dataset. The grouped CV above is the held-out separation estimate.

In [ ]:
post_direction_path = post_dir / "s2_direction_block30.pt"
if post_activation_path.exists() and post_direction_path.exists():
    post_x = post_record["states"].float()
    post_direction_record = torch.load(post_direction_path, map_location="cpu", weights_only=False)
    post_direction = post_direction_record["denoised_direction"].float()
    assert post_x.shape == x.shape and post_direction.shape == base_direction.shape
    print("Cosine(Base, post-trained) at block 30:",
          round(F.cosine_similarity(base_direction, post_direction, dim=0).item(), 4))
    transfer = []
    for checkpoint, activations in (("Base", x), ("Post-trained", post_x)):
        for direction_name, direction in (("Base", base_direction), ("Post-trained", post_direction)):
            auc = roc_auc_score(y, (activations[:, BLOCK] @ direction).numpy())
            transfer.append({"checkpoint": checkpoint, "direction": direction_name,
                             "source_in_sample_auc": auc})
    transfer = pd.DataFrame(transfer)
    transfer.to_csv(run_dir / "source_direction_transfer.csv", index=False)
    print(transfer.pivot(index="checkpoint", columns="direction",
                         values="source_in_sample_auc").round(4))
else:
    print("Post-trained run artifacts absent. Re-run this cell when available.")

## Record interpretation before fresh-prompt confirmation

Fill after execution: Base fixed-block AUC, best Base layer (descriptive), 2×2 alignment effect and interval, cross-check with post-trained results, deviations, and surprising category/control behavior. These source-prompt checks are **not** an independent confirmation of the frozen motif. Confirmation requires a fresh matched prompt set designed without using these Base outputs.